# Kaggle Titantic Prediction Model: Random Forest
Conforms to the "Titantic Tutorial" notebook on Kaggle, as of April 14, 2026 (https://www.kaggle.com/code/alexisbcook/titanic-tutorial). This is run locally.

The competition is simple: use machine learning to create a model that predicts which passengers survived the Titanic shipwreck. In this challenge, we ask you to build a predictive model that answers the question: “what sorts of people were more likely to survive?” using passenger data (ie name, age, gender, socio-economic class, etc).


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load in 

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the "../input/" directory.
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Any results you write to the current directory are saved as output.

## Load the Data
Loads the data from the train.csv and test.csv files in the same directory and output a preview

In [3]:
train_data = pd.read_csv("train.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
test_data = pd.read_csv("test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


## Explore Patterns
Test some very basic (and unlikely) patterns to get a feel for the data. Then, move into more complex patterns to train the model.

In [5]:
# Pattern 1: assumes that all female passengers survived (and all male passengers died.)

women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

% of women who survived: 0.7420382165605095


In [7]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

% of men who survived: 0.18890814558058924


## Construct a Model: Version 1
This will be a **random forest model**. Each tree will individually consider each passenger's data and vote on whether the individual survived. Then, the random forest model makes a democratic decision: the outcome with the most votes wins!

The code cell below constructs a model that makes a prediction based on the columns ("Pclass", "Sex", "SibSp", and "Parch"). It then saves these predictions in a CSV file called **submission.csv**, to be uploaded to Kaggle.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# feature engineering
def engineer_features(df):
    '''# OBSERVATIONS ON CORRELATION BETWEEN NO CABIN DATA/FARE/PASSENGER CLASS: 
       # it seemed that a lack of cabin data is highly correlated with lower-class passengers (high value for Deck_N).
       # it seemed that a lack of cabin data is also correlated with lower Fare value and decks B and C are correlated with
       # higher fare values (~-0.5 vs 0.4-0.5.)'''
    df = df.copy()

    # fill missing values
    df['Age'].fillna(df['Age'].median(), inplace=True)
    df['Fare'].fillna(df['Fare'].median(), inplace=True)
    df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

    # add four new features
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1 # Parch is the number of parents/children traveling abroad
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace(['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare') # list of all possible titles
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})

    # cabin column engineering
    cabin_only = df[["Cabin"]].copy()
    cabin_only["Cabin_Data"] = cabin_only["Cabin"].isnull().apply(lambda x: not x)
    cabin_only["Deck"] = cabin_only["Cabin"].str.slice(0,1)
    cabin_only["Room"] = cabin_only["Cabin"].str.slice(1,5).str.extract("([0-9]+)", expand=False).astype("float")
    cabin_only[cabin_only["Cabin_Data"]]

    cabin_only.drop(["Cabin", "Cabin_Data"], axis=1, inplace=True, errors="ignore")
    cabin_only["Deck"] = cabin_only["Deck"].fillna("N") # represents the lack of data for a deck
    cabin_only["Room"] = cabin_only["Room"].fillna(cabin_only["Room"].mean())

    def encode_column(df, label, drop_col=False):
        '''Uses one-hot encoding on the Deck column, returning a pandas dataframe with the encoding'''
        one_hot = pd.get_dummies(df[label], prefix=label)
        if drop_col:
            df = df.drop(label, axis=1)
        df = df.join(one_hot)
        return df
    
    def one_hot(df, labels, drop_col=False):
        '''One hot encode a list of columns, for example putting together several
        one-hot encoded Deck columns'''
        for label in labels:
            df = encode_column(df, label, drop_col)
        return df
    
    cabin_only = one_hot(cabin_only, ["Deck"], drop_col=True) # one-hot encode all the Deck columns
    #print(cabin_only.head()) - was used for testing
    for column in cabin_only.columns: # attempt to discover correlations between Cabin data and survival data
        df[column] = cabin_only[column]

    df.drop(["Ticket", "Cabin"], axis=1, inplace=True)
    corr = df.corr(method='pearson', numeric_only=True)
    #print(corr["Fare"].sort_values(ascending=False)) -  was used for testing

    return df

train_data = engineer_features(train_data) # create the new data for the model to train on
test_data = engineer_features(test_data)
print(test_data.head())

# define features and build X / X_test together
deck_columns = [col for col in train_data.columns if col.startswith("Deck_") and col in test_data.columns] # added the test data to this iteration to fix the Deck_T error
features = ["Pclass", "Sex", "Age", "Fare", "SibSp", "Parch", "Embarked", "FamilySize", "IsAlone", "Title", "Room"] + deck_columns 
X = pd.get_dummies(train_data[features])
X_test = pd.get_dummies(test_data[features])

# Force X_test to have the exact same columns as X/lm
# This adds missing columns (as 0) and removes extra ones found only in test
X_test = X_test.reindex(columns=X.columns, fill_value=0)
y = train_data["Survived"]

model = RandomForestClassifier(n_estimators=400, max_depth=None, random_state=42, max_leaf_nodes = 5000, max_features=3, min_samples_leaf=1, min_samples_split=10, bootstrap=True)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission_v7.csv', index=False) # this exports the predictions of the final model. to get metrics, run classification report in the next cell, calling 'model'.
print("Your submission was successfully saved!")

C:\Users\car-m\AppData\Local\Temp\ipykernel_7612\951966694.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Age'].fillna(df['Age'].median(), inplace=True)
C:\Users\car-m\AppData\Local\Temp\ipykernel_7612\951966694.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For exa

   PassengerId  Pclass                                          Name     Sex  \
0          892       3                              Kelly, Mr. James    male   
1          893       3              Wilkes, Mrs. James (Ellen Needs)  female   
2          894       2                     Myles, Mr. Thomas Francis    male   
3          895       3                              Wirz, Mr. Albert    male   
4          896       3  Hirvonen, Mrs. Alexander (Helga E Lindqvist)  female   

    Age  SibSp  Parch     Fare Embarked  FamilySize  ...  Title       Room  \
0  34.5      0      0   7.8292        Q           1  ...     Mr  47.651685   
1  47.0      1      0   7.0000        S           2  ...    Mrs  47.651685   
2  62.0      0      0   9.6875        Q           1  ...     Mr  47.651685   
3  27.0      0      0   8.6625        S           1  ...     Mr  47.651685   
4  22.0      1      1  12.2875        S           3  ...    Mrs  47.651685   

   Deck_A  Deck_B  Deck_C  Deck_D  Deck_E  Deck_F 

### Getting Data on the Model's Performance

In [ ]:
# this value is the mean absolute error of the model's predictions on the training data, which is not a good metric for this classification problem but is included here for demonstration purposes.
# it typically prints suspiciously high values.
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import classification_report

predicted_survival = model.predict(X)
mean_absolute_error(y, predicted_survival)
print(classification_report(y, predicted_survival))



              precision    recall  f1-score   support

           0       0.89      0.95      0.92       549
           1       0.91      0.81      0.86       342

    accuracy                           0.90       891
   macro avg       0.90      0.88      0.89       891
weighted avg       0.90      0.90      0.90       891



In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, accuracy_score

# split data into training and validation data, for both features and target
# The split is based on a random number generator. Supplying a numeric value to
# the random_state argument guarantees we get the same split every time we
# run this script.

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 42)
eval_model = RandomForestClassifier(n_estimators=400, max_depth=None, random_state=42, max_leaf_nodes = 5000, max_features=3, min_samples_leaf=1, min_samples_split=10, bootstrap=True) # model for evaluation ONLY
eval_model.fit(train_X, train_y) # fit the evaluation model
val_predictions = eval_model.predict(val_X)
print(classification_report(val_y, val_predictions)) # your actual score
probs = eval_model.predict_proba(val_X)[:, 1] # for the eval model only
# roc_curve(val_y, probs)
roc_auc_score(val_y, probs)

# Define FINAL model
final_model = RandomForestClassifier(n_estimators=400, max_depth=None, random_state=42, max_leaf_nodes = 5000, max_features=3, min_samples_leaf=1, min_samples_split=10, bootstrap=True)
# Fit model
final_model.fit(X, y) # fit on all the data, not just the training data

final_predictions = final_model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': final_predictions})
output.to_csv('submission_v9.csv', index=False)
print("Your submission was successfully saved!")


              precision    recall  f1-score   support

           0       0.86      0.88      0.87       134
           1       0.81      0.79      0.80        89

    accuracy                           0.84       223
   macro avg       0.84      0.83      0.84       223
weighted avg       0.84      0.84      0.84       223

Your submission was successfully saved!


### Fine-Tuning the Model
Used GridSearchCV and RandomizedSearchCV to find the optimal parameters for the model.

In [ ]:
# Compare mean absolute error scores when we change the value for max_leaf_nodes
# max_leaf_nodes allows us to control underfitting vs overfitting

# from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# using RandomizedSearchCV to find the best parameters for the model using k-fold cross-validation
param_grid = {
    'n_estimators': [200, 300, 400], 
    'max_depth': [None, 5, 10, 20], 
    'max_features': [3, 6], 
    'min_samples_split': [2, 5, 10], 
    'min_samples_leaf': [1, 2, 5]
}

random_search = RandomizedSearchCV(RandomForestClassifier(), param_grid, cv=5, n_jobs = -1)
random_search.fit(train_X, train_y)
print(f"Best Params: {random_search.best_params_}")

Best Params: {'n_estimators': 400, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 3, 'max_depth': None}
